### Create work-funder associations from PubMed metadata

Extracts the `funders` array from **PubMed** works in `locations_mapped` (parsed from the PubMed `<GrantList>` by `notebooks/ingest/PubMed.py`), resolves the `<Agency>` string to a funder **by name** (PubMed grants carry no funder DOI), and writes a junction table analogous to `crossref_work_funders` / `openalex.mid.work_funder`.

Why this exists: many funders require grantees to assert grant↔output linkages in PubMed/PMC, so these linkages exist in PubMed even when Crossref has no funding metadata. We were parsing `<GrantList>` into `locations_mapped(provenance='pubmed')` but never consuming it. (Note: for non-NIH funders PubMed GrantList coverage is thin — PMC `<funding-group>` is the richer source, handled by a separate PMC leg once that parser exists.)

Name resolution reuses `openalex.common.funder_names_keep` (the curated name→id allowlist the fulltext leg uses). Agency strings that don't exactly match an allowlisted name are dropped (INNER JOIN) — extend `funder_names_keep` with PubMed `<Agency>` aliases to raise coverage.

This table feeds into:
- **CreateWorksEnriched** (`from_pubmed` funder union leg)
- **openalex_awards_raw** (provenance `pubmed_work_funders`, priority 2)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pubmed_work_funders
USING delta
AS
WITH exploded AS (
  SELECT
    lm.work_id,
    f.name AS funder_name,
    f.awards AS award_ids
  FROM openalex.works.locations_mapped lm
  LATERAL VIEW EXPLODE(funders) AS f
  WHERE lm.provenance = 'pubmed'
    AND lm.work_id IS NOT NULL
    AND f.name IS NOT NULL
),
matched AS (
  -- Resolve the PubMed <Agency> string to a funder by NAME (PubMed grants carry no
  -- funder DOI), via the curated name->id allowlist used by the fulltext leg.
  -- funder_names_keep.id is the 'F<digits>' funder id; strip the leading 'F' to the
  -- numeric funder_id used by crossref_work_funders / mid.funder.
  SELECT
    e.work_id,
    CAST(SUBSTRING(fnk.id, 2) AS BIGINT) AS funder_id,
    e.award_ids
  FROM exploded e
  JOIN openalex.common.funder_names_keep fnk ON fnk.name = e.funder_name
),
-- Explode each award_id so we can apply is_usable_award_id per-element.
-- Databricks rejects UDF calls inside higher-order functions (FILTER, TRANSFORM),
-- so we can't do FILTER(..., x -> is_usable_award_id(x)). OUTER EXPLODE preserves
-- rows where award_ids is NULL/empty, so the work-funder link is kept even when no
-- specific awards survive.
flattened AS (
  SELECT
    work_id,
    funder_id,
    CASE WHEN openalex.common.is_usable_award_id(aid) THEN aid END AS aid
  FROM matched
  LATERAL VIEW OUTER EXPLODE(award_ids) AS aid
)
-- Re-aggregate one row per (work_id, funder_id). COLLECT_LIST ignores NULLs, so the
-- CASE WHEN drops junk elements naturally. ARRAY_DISTINCT preserves dedup semantics.
SELECT
  work_id,
  funder_id,
  ARRAY_DISTINCT(COLLECT_LIST(aid)) AS award_ids
FROM flattened
GROUP BY work_id, funder_id

### Sanity checks

In [ ]:
%sql
-- Row counts and uniqueness
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id)) AS distinct_keys,
  COUNT(DISTINCT work_id) AS distinct_works,
  COUNT(DISTINCT funder_id) AS distinct_funders,
  COUNT(CASE WHEN SIZE(award_ids) > 0 THEN 1 END) AS rows_with_awards
FROM openalex.awards.pubmed_work_funders

In [ ]:
%sql
-- Verify composite key uniqueness (should return 0)
SELECT COUNT(*) AS duplicate_keys
FROM (
  SELECT work_id, funder_id, COUNT(*) AS n
  FROM openalex.awards.pubmed_work_funders
  GROUP BY work_id, funder_id
  HAVING n > 1
)

In [ ]:
%sql
-- Coverage spot-check: AHA (F4320306230). How many PubMed work-funder links did we
-- resolve, and how many carry an award id? Useful to confirm funder_names_keep has the
-- AHA <Agency> aliases before trusting the leg.
SELECT
  COUNT(*) AS aha_links,
  COUNT(CASE WHEN SIZE(award_ids) > 0 THEN 1 END) AS aha_links_with_award
FROM openalex.awards.pubmed_work_funders
WHERE funder_id = 4320306230

### Insert awards into `openalex_awards_raw`

Follows the same pattern as `CreateCrossrefWorkFunders`: explode award_ids, generate xxhash64-based IDs, insert with provenance `pubmed_work_funders` and priority 2 (peer of the crossref work-funder leg; dedup in `CreateAwards` keys on award id ordered by priority, and the DELETE keys on provenance+priority).

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW pubmed_work_funder_awards AS
WITH exploded AS (
  SELECT DISTINCT
    pwf.funder_id,
    LOWER(award_id) AS normalized_award_id,
    award_id AS funder_award_id
  FROM openalex.awards.pubmed_work_funders pwf
  LATERAL VIEW EXPLODE(award_ids) AS award_id
  WHERE SIZE(award_ids) > 0
),
funders AS (
  SELECT funder_id, display_name, ror_id, doi
  FROM openalex.mid.funder
)
SELECT
  ABS(XXHASH64(CONCAT(f.funder_id, ':', e.normalized_award_id))) % 9000000000 AS id,
  CAST(NULL AS STRING) AS display_name,
  CAST(NULL AS STRING) AS description,
  f.funder_id,
  e.funder_award_id,
  CAST(NULL AS DOUBLE) AS amount,
  CAST(NULL AS STRING) AS currency,
  STRUCT(
    CONCAT('https://openalex.org/F', f.funder_id) AS id,
    f.display_name,
    f.ror_id,
    f.doi
  ) AS funder,
  CAST(NULL AS STRING) AS funding_type,
  CAST(NULL AS STRING) AS funder_scheme,
  'pubmed_work_funders' AS provenance,
  CAST(NULL AS DATE) AS start_date,
  CAST(NULL AS DATE) AS end_date,
  CAST(NULL AS INT) AS start_year,
  CAST(NULL AS INT) AS end_year,
  CAST(NULL AS STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >) AS lead_investigator,
  CAST(NULL AS STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >) AS co_lead_investigator,
  CAST(NULL AS ARRAY<STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >>) AS investigators,
  CAST(NULL AS STRING) AS landing_page_url,
  openalex.common.extract_grant_doi(e.funder_award_id) AS doi,
  CONCAT('https://api.openalex.org/works?filter=awards.id:G', ABS(XXHASH64(CONCAT(f.funder_id, ':', e.normalized_award_id))) % 9000000000) AS works_api_url,
  CURRENT_TIMESTAMP() AS created_date,
  CURRENT_TIMESTAMP() AS updated_date
FROM exploded e
JOIN funders f ON f.funder_id = e.funder_id

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pubmed_work_funders' AND priority = 2

In [ ]:
%sql
INSERT INTO openalex.awards.openalex_awards_raw
SELECT *, 2 AS priority
FROM pubmed_work_funder_awards